# Disease Label Definitions

**Dataset:** `nhanes_merged_adults_final.csv`
**Population:** NHANES adults (n = 7,437)
**Total disease labels:** 23

This notebook documents the exact derivation formula for every binary disease column in the dataset.
Each label is `1` = condition present, `0` = absent.
Source column names refer to the **renamed columns** as they appear in the merged file (e.g. `mcq053___taking_treatment_for_anemia/past_3_mos`), with the original NHANES variable code noted alongside.

---

| # | Label | Type | Primary source |
|---|-------|------|----------------|
| 1 | `anemia` | Questionnaire | MCQ053 |
| 2 | `diabetes` | Questionnaire | DIQ010, DIQ160 |
| 3 | `thyroid` | Questionnaire | MCQ170M |
| 4 | `sleep_disorder` | Questionnaire + RXD | SLQ040, SLQ050, RXDRSD1-3 |
| 5 | `kidney` | Questionnaire + Lab | KIQ022, LBXSCR |
| 6 | `hepatitis_bc` | Questionnaire | HEQ010, HEQ030 |
| 7 | `liver` | Questionnaire | MCQ510A–F |
| 8 | `heart_failure` | Questionnaire | MCQ160B |
| 9 | `coronary_heart` | Questionnaire | MCQ160C |
| 10 | `emphysema_lungs` | Questionnaire | MCQ160P |
| 11 | `high_blood_pressure` | Questionnaire | BPQ020 |
| 12 | `high_cholesterol` | Questionnaire | BPQ080 |
| 13 | `menopause` | Questionnaire | RHQ305, RHD043, RHQ031 |
| 14 | `overweight` | Questionnaire | MCQ080 |
| 15 | `alcohol` | Questionnaire | ALQ151 |
| 16 | `iron_deficiency` | Lab | LBXFER |
| 17 | `hepatic_insufficiency` | Lab | LBXSATSI, LBXSASSI, LBXSGTSI |
| 18 | `electrolyte_imbalance` | Lab | LBXSNASI, LBXSKSI, LBXSCA |
| 19 | `infection_inflammation` | Lab | LBXHSCRP, LBXWBCSI, LBXNEPCT, LBXLYPCT (multi-marker score) |
| 20 | `CFS_suspect` | Questionnaire + Lab | Multi-criterion |
| 21 | `myalgia` | RXD | RXDRSD1-3 keyword |
| 22 | `anxiety` | RXD | RXDRSD1-3 keyword |
| 23 | `depression` | RXD | RXDRSD1-3 keyword |

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/nhanes_merged_adults_final.csv', low_memory=False)

ALL_23 = [
    'anemia','diabetes','thyroid','sleep_disorder','kidney',
    'hepatitis_bc','liver','heart_failure','coronary_heart',
    'emphysema_lungs','high_blood_pressure','high_cholesterol',
    'menopause','overweight','alcohol',
    'iron_deficiency','hepatic_insufficiency','electrolyte_imbalance',
    'infection_inflammation','CFS_suspect',
    'myalgia','anxiety','depression',
]

print(f'Dataset: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
print(f'All 23 labels present: {all(c in df.columns for c in ALL_23)}')
print(f'rxd_disease_list present: {"rxd_disease_list" in df.columns}')


Dataset: 7,437 rows x 877 columns
All 23 labels present: True
rxd_disease_list present: True


---
## 1 · anemia

**Definition:** Person was taking treatment for anemia in the past 3 months.

| NHANES variable | Renamed column | Question text |
|-----------------|----------------|---------------|
| MCQ053 | `mcq053___taking_treatment_for_anemia/past_3_mos` | *"Are you now taking treatment for anemia?"* |

**Formula:**
```
anemia = 1   if  MCQ053 == 1
       = 0   otherwise
```

**MCQ053 encoding:** 1 = Yes · 2 = No · 9 = Don't know  
**Notes:** Captures medically confirmed and actively treated anaemia. Does not capture untreated or subclinical iron-deficiency anaemia (→ see `iron_deficiency`).

In [2]:
col = 'mcq053___taking_treatment_for_anemia/past_3_mos'
print('MCQ053 value counts:')
print(df[col].value_counts(dropna=False).sort_index().rename({1:'1=Yes',2:'2=No',9:'9=DK',np.nan:'NaN'}).to_string())
print(f'\nanemia  →  1: {int(df["anemia"].sum()):,}   0: {int((df["anemia"]==0).sum()):,}')

MCQ053 value counts:
mcq053___taking_treatment_for_anemia/past_3_mos
1=Yes     358
2=No     7068
9=DK       11

anemia  →  1: 358   0: 7,079


---
## 2 · diabetes

**Definition:** Doctor-diagnosed diabetes OR borderline/pre-diabetes.

| NHANES variable | Renamed column | Question text |
|-----------------|----------------|---------------|
| DIQ010 | `diq010___doctor_told_you_have_diabetes` | *"Has a doctor told you that you have diabetes?"* |
| DIQ160 | `diq160___ever_told_you_have_prediabetes` | *"Have you ever been told you have pre-diabetes?"* |

**Formula:**
```
diabetes = 1   if  DIQ010 ∈ {1, 3}        # 1=Yes  3=Borderline
                OR DIQ160 == 1             # pre-diabetes
         = 0   otherwise
```

**DIQ010 encoding:** 1 = Yes · 2 = No · 3 = Borderline · 9 = Don't know  
**DIQ160 encoding:** 1 = Yes · 2 = No · 9 = Don't know  
**Notes:** DIQ010 = 3 (borderline) and DIQ160 = 1 (pre-diabetes) are included to capture the full glucose-dysregulation spectrum relevant to clinical risk.

In [3]:
for col, name in [('diq010___doctor_told_you_have_diabetes','DIQ010'),
                  ('diq160___ever_told_you_have_prediabetes','DIQ160')]:
    print(f'{name} value counts:')
    print(df[col].value_counts(dropna=False).sort_index().to_string())
    print()
print(f'diabetes  →  1: {int(df["diabetes"].sum()):,}   0: {int((df["diabetes"]==0).sum()):,}')

DIQ010 value counts:
diq010___doctor_told_you_have_diabetes
1.0     777
2.0    6471
3.0     186
9.0       3

DIQ160 value counts:
diq160___ever_told_you_have_prediabetes
1.0     682
2.0    5781
9.0      11
NaN     963

diabetes  →  1: 1,645   0: 5,792


---
## 3 · thyroid

**Definition:** Currently has a thyroid problem (not historical only).

| NHANES variable | Renamed column | Question text |
|-----------------|----------------|---------------|
| MCQ170M | `mcq170m___still_have_thyroid_problem` | *"Do you still have a thyroid problem?"* |

**Formula:**
```
thyroid = 1   if  MCQ170M == 1
        = 0   otherwise
```

**MCQ170M encoding:** 1 = Yes · 2 = No · 9 = Don't know  
**Notes:** MCQ170M is only asked of respondents who first answered MCQ160M = 1 (ever told you had thyroid problem). NaN for respondents never diagnosed — these map to 0.

In [4]:
col = 'mcq170m___still_have_thyroid_problem'
print('MCQ170M value counts:')
print(df[col].value_counts(dropna=False).sort_index().to_string())
print(f'\nthyroid  →  1: {int(df["thyroid"].sum()):,}   0: {int((df["thyroid"]==0).sum()):,}')

MCQ170M value counts:
mcq170m___still_have_thyroid_problem
1.0     462
2.0     149
9.0      27
NaN    6799

thyroid  →  1: 462   0: 6,975


---
## 4 · sleep_disorder

**Definition:** Clinically significant sleep-disordered breathing OR doctor-acknowledged sleep trouble
**— enriched** with medication prescription data (RXDRSD columns).

| NHANES variable | Renamed column | Question text |
|-----------------|----------------|---------------|
| SLQ040 | `slq040___how_often_do_you_snort_or_stop_breathing` | *"How often do you snort, gasp, or stop breathing?"* |
| SLQ050 | `slq050___ever_told_doctor_had_trouble_sleeping?` | *"Have you ever told a doctor that you have trouble sleeping?"* |
| RXDRSD1-3 | `rxd_disease_list` | Medication disease indications (P_RXQ_RX.xpt) |

**Formula:**
```
sleep_disorder = 1   if  SLQ040 ∈ {2, 3}                   # Occasionally or Frequently
                      OR SLQ050 == 1                         # Told doctor about sleep trouble
                      OR rxd_disease_list contains "Insomnia"
                      OR rxd_disease_list contains "Sleep disorder"
               = 0   otherwise
```

**SLQ040 encoding:** 0=Never · 1=Rarely · 2=Occasionally · 3=Frequently · 4=Almost always
**SLQ050 encoding:** 1=Yes · 2=No
**Notes:** SLQ120 (daytime sleepiness) was specified in the original definition but is **not present** in this dataset.
The RXD enrichment added **19** additional positives (people prescribed sleep medication whose questionnaire responses were borderline/missing).


In [5]:
for col, name in [('slq040___how_often_do_you_snort_or_stop_breathing','SLQ040'),
                  ('slq050___ever_told_doctor_had_trouble_sleeping?','SLQ050')]:
    print(f'{name} value counts:')
    print(df[col].value_counts(dropna=False).sort_index().to_string())
    print()

# RXD enrichment check
rxd_sleep = df['rxd_disease_list'].str.contains('Insomnia|Sleep disorder', case=False, na=False)
print(f'People with Insomnia/Sleep disorder in RXD: {rxd_sleep.sum():,}')
print(f'sleep_disorder  ->  1: {int(df["sleep_disorder"].sum()):,}   0: {int((df["sleep_disorder"]==0).sum()):,}')


SLQ040 value counts:
slq040___how_often_do_you_snort_or_stop_breathing
0.0    5387
1.0     922
2.0     422
3.0     380
7.0       3
9.0     323

SLQ050 value counts:
slq050___ever_told_doctor_had_trouble_sleeping?
1.0    1965
2.0    5466
7.0       1
9.0       5

People with Insomnia/Sleep disorder in RXD: 275
sleep_disorder  ->  1: 2,403   0: 5,034


---
## 5 · kidney

**Definition:** Doctor-diagnosed weak/failing kidneys OR elevated serum creatinine by sex-adjusted threshold.

| NHANES variable | Renamed column | Source |
|-----------------|----------------|--------|
| KIQ022 | `kiq022___ever_told_you_had_weak/failing_kidneys?` | Questionnaire |
| LBXSCR | `LBXSCR_creatinine_refrigerated_serum_mg_dl` | Lab |

**Formula:**
```
kidney = 1   if  KIQ022 == 1                     # doctor diagnosis
              OR ( gender == Female  AND  LBXSCR > 1.2 mg/dL )
              OR ( gender == Male    AND  LBXSCR > 1.4 mg/dL )
       = 0   otherwise
```

**KIQ022 encoding:** 1=Yes · 2=No · 9=Don't know  
**Thresholds:** LBXSCR > 1.2 (F) / > 1.4 (M) — standard upper-normal limits for serum creatinine (mg/dL).  
**Notes:** Lab criterion catches undiagnosed CKD/reduced GFR not yet formally diagnosed.

In [6]:
kiq_col = 'kiq022___ever_told_you_had_weak/failing_kidneys?'
scr_col = 'LBXSCR_creatinine_refrigerated_serum_mg_dl'
print('KIQ022 value counts:')
print(df[kiq_col].value_counts(dropna=False).sort_index().to_string())
print(f'\nLBXSCR  n={df[scr_col].notna().sum():,}  median={df[scr_col].median():.2f} mg/dL')
is_female = df['gender'] == 'Female'
lab_flag  = ((is_female & (df[scr_col] > 1.2)) | (~is_female & (df[scr_col] > 1.4)))
print(f'  Elevated creatinine (lab criterion alone): {lab_flag.sum():,}')
print(f'  KIQ022==1 only: {(df[kiq_col]==1).sum():,}')
print(f'\nkidney  →  1: {int(df["kidney"].sum()):,}   0: {int((df["kidney"]==0).sum()):,}')

KIQ022 value counts:
kiq022___ever_told_you_had_weak/failing_kidneys?
1.0     186
2.0    6785
9.0       5
NaN     461

LBXSCR  n=6,389  median=0.81 mg/dL
  Elevated creatinine (lab criterion alone): 134
  KIQ022==1 only: 186

kidney  →  1: 259   0: 7,178


---
## 6 · hepatitis_bc

**Definition:** Ever diagnosed with Hepatitis B or Hepatitis C.

| NHANES variable | Renamed column | Question text |
|-----------------|----------------|---------------|
| HEQ010 | `heq010___ever_told_you_have_hepatitis_b?` | *"Has a doctor ever told you that you have Hepatitis B?"* |
| HEQ030 | `heq030___ever_told_you_have_hepatitis_c?` | *"Has a doctor ever told you that you have Hepatitis C?"* |

**Formula:**
```
hepatitis_bc = 1   if  HEQ010 == 1   # Hepatitis B
                    OR HEQ030 == 1   # Hepatitis C
             = 0   otherwise
```

**Encoding:** 1=Yes · 2=No · 7=Refused · 9=Don't know

In [7]:
for col, name in [('heq010___ever_told_you_have_hepatitis_b?','HEQ010'),
                  ('heq030___ever_told_you_have_hepatitis_c?','HEQ030')]:
    print(f'{name}: {df[col].value_counts(dropna=False).sort_index().to_dict()}')
print(f'\nhepatitis_bc  →  1: {int(df["hepatitis_bc"].sum()):,}   0: {int((df["hepatitis_bc"]==0).sum()):,}')

HEQ010: {1.0: 73, 2.0: 7334, 7.0: 1, 9.0: 29}
HEQ030: {1.0: 128, 2.0: 7281, 9.0: 28}

hepatitis_bc  →  1: 192   0: 7,245


---
## 7 · liver

**Definition:** Doctor-confirmed liver condition of any sub-type.

| NHANES variable | Renamed column | Sub-type |
|-----------------|----------------|----------|
| MCQ510A | `mcq510a___liver_condition_fatty_liver` | Fatty liver (coded 1) |
| MCQ510B | `mcq510b___liver_condition_non_alcoholic_fatty_liver` | NAFLD (coded 2) |
| MCQ510C | `mcq510c___liver_condition_alcoholic_liver_disease` | Alcoholic (coded 3) |
| MCQ510D | `mcq510d___liver_condition_hepatitis` | Liver hepatitis (coded 4) |
| MCQ510E | `mcq510e___liver_condition_autoimmune` | Autoimmune (coded 5) |
| MCQ510F | `mcq510f___liver_condition_other` | Other (coded 6) |

**Formula:**
```
liver = 1   if  MCQ510A == 1    # each column uses its own code value
             OR MCQ510B == 2
             OR MCQ510C == 3
             OR MCQ510D == 4
             OR MCQ510E == 5
             OR MCQ510F == 6
      = 0   otherwise
```

**Notes:** These are response-option columns — each is only non-null when the respondent was routed to the liver sub-type question (i.e., answered MCQ160L = 1 *or* MCQ500 = 1 first). NaN = not applicable (no liver condition reported), mapped to 0.

In [8]:
liver_cols = [
    ('mcq510a___liver_condition_fatty_liver', 1),
    ('mcq510b___liver_condition_non_alcoholic_fatty_liver', 2),
    ('mcq510c___liver_condition_alcoholic_liver_disease', 3),
    ('mcq510d___liver_condition_hepatitis', 4),
    ('mcq510e___liver_condition_autoimmune', 5),
    ('mcq510f___liver_condition_other', 6),
]
for col, code in liver_cols:
    n = (df[col] == code).sum()
    print(f'  {col.split("___")[0].upper()} == {code} : {n:>4} cases')
print(f'\nliver  →  1: {int(df["liver"].sum()):,}   0: {int((df["liver"]==0).sum()):,}')

  MCQ510A == 1 :  147 cases
  MCQ510B == 2 :    6 cases
  MCQ510C == 3 :   22 cases
  MCQ510D == 4 :   59 cases
  MCQ510E == 5 :   13 cases
  MCQ510F == 6 :   73 cases

liver  →  1: 302   0: 7,135


---
## 8 · heart_failure

| NHANES variable | Renamed column | Question text |
|-----------------|----------------|---------------|
| MCQ160B | `mcq160b___ever_told_you_had_congestive_heart_failure` | *"Has a doctor told you that you have congestive heart failure?"* |

**Formula:**
```
heart_failure = 1   if  MCQ160B == 1
              = 0   otherwise
```

In [9]:
col = 'mcq160b___ever_told_you_had_congestive_heart_failure'
print(f'MCQ160B: {df[col].value_counts(dropna=False).sort_index().to_dict()}')
print(f'heart_failure  →  1: {int(df["heart_failure"].sum()):,}   0: {int((df["heart_failure"]==0).sum()):,}')

MCQ160B: {1.0: 149, 2.0: 6816, 9.0: 11, nan: 461}
heart_failure  →  1: 149   0: 7,288


---
## 9 · coronary_heart

| NHANES variable | Renamed column | Question text |
|-----------------|----------------|---------------|
| MCQ160C | `mcq160c___ever_told_you_had_coronary_heart_disease` | *"Has a doctor told you that you have coronary heart disease?"* |

**Formula:**
```
coronary_heart = 1   if  MCQ160C == 1
               = 0   otherwise
```

In [10]:
col = 'mcq160c___ever_told_you_had_coronary_heart_disease'
print(f'MCQ160C: {df[col].value_counts(dropna=False).sort_index().to_dict()}')
print(f'coronary_heart  →  1: {int(df["coronary_heart"].sum()):,}   0: {int((df["coronary_heart"]==0).sum()):,}')

MCQ160C: {1.0: 154, 2.0: 6811, 9.0: 11, nan: 461}
coronary_heart  →  1: 154   0: 7,283


---
## 10 · emphysema_lungs

| NHANES variable | Renamed column | Question text |
|-----------------|----------------|---------------|
| MCQ160P | `mcq160p___ever_told_you_had_copd_emphysema` | *"Has a doctor told you that you have COPD or emphysema?"* |

**Formula:**
```
emphysema_lungs = 1   if  MCQ160P == 1
                = 0   otherwise
```

In [11]:
col = 'mcq160p___ever_told_you_had_copd_emphysema'
print(f'MCQ160P: {df[col].value_counts(dropna=False).sort_index().to_dict()}')
print(f'emphysema_lungs  →  1: {int(df["emphysema_lungs"].sum()):,}   0: {int((df["emphysema_lungs"]==0).sum()):,}')

MCQ160P: {1.0: 477, 2.0: 6491, 9.0: 8, nan: 461}
emphysema_lungs  →  1: 477   0: 6,960


---
## 11 · high_blood_pressure

| NHANES variable | Renamed column | Question text |
|-----------------|----------------|---------------|
| BPQ020 | `bpq020___ever_told_you_had_high_blood_pressure` | *"Has a doctor told you that you have high blood pressure?"* |

**Formula:**
```
high_blood_pressure = 1   if  BPQ020 == 1
                    = 0   otherwise
```

In [12]:
col = 'bpq020___ever_told_you_had_high_blood_pressure'
print(f'BPQ020: {df[col].value_counts(dropna=False).sort_index().to_dict()}')
print(f'high_blood_pressure  →  1: {int(df["high_blood_pressure"].sum()):,}   0: {int((df["high_blood_pressure"]==0).sum()):,}')

BPQ020: {1.0: 2130, 2.0: 5299, 9.0: 8}
high_blood_pressure  →  1: 2,130   0: 5,307


---
## 12 · high_cholesterol

| NHANES variable | Renamed column | Question text |
|-----------------|----------------|---------------|
| BPQ080 | `bpq080___doctor_told_you___high_cholesterol_level` | *"Has a doctor told you that your cholesterol level is high?"* |

**Formula:**
```
high_cholesterol = 1   if  BPQ080 == 1
                 = 0   otherwise
```

In [13]:
col = 'bpq080___doctor_told_you___high_cholesterol_level'
print(f'BPQ080: {df[col].value_counts(dropna=False).sort_index().to_dict()}')
print(f'high_cholesterol  →  1: {int(df["high_cholesterol"].sum()):,}   0: {int((df["high_cholesterol"]==0).sum()):,}')

BPQ080: {1.0: 1977, 2.0: 5423, 9.0: 37}
high_cholesterol  →  1: 1,977   0: 5,460


---
## 13 · menopause

**Definition:** Female has reached menopause — via surgical removal of both ovaries, cessation of periods due to menopause, or natural cessation in women over 40.

| NHANES variable | Renamed column | Question text |
|-----------------|----------------|---------------|
| RHQ305 | `rhq305___had_both_ovaries_removed?` | *"Had both ovaries removed?"* |
| RHD043 | `rhd043___reason_not_having_regular_periods` | *"Reason for not having regular periods"* |
| RHQ031 | `rhq031___had_regular_periods_in_past_12_months` | *"Had regular periods in past 12 months?"* |
| RIDAGEYR | `age_years` | Age in years |

**Formula:**
```
menopause = 1   if  RHQ305 == 1                          # surgical menopause (oophorectomy)
                 OR RHD043 == 7                           # reason = menopause
                 OR ( RHQ031 == 2  AND  age_years > 40 ) # no periods + age > 40
          = 0   otherwise
```

**RHQ305 encoding:** 1=Yes · 2=No  
**RHD043 encoding (selected):** 7=Menopause / hysterectomy · 3=Pregnant  
**RHQ031 encoding:** 1=Yes · 2=No (no periods in past 12 months)  
**Notes:** Only applied to female respondents (`gender == Female`). Male respondents are 0 by construction (NaN from female-only questions → 0).

In [14]:
for col, name in [('rhq305___had_both_ovaries_removed?','RHQ305'),
                  ('rhd043___reason_not_having_regular_periods','RHD043'),
                  ('rhq031___had_regular_periods_in_past_12_months','RHQ031')]:
    print(f'{name}: {df[col].value_counts(dropna=False).sort_index().to_dict()}')
print(f'\nmenopause  →  1: {int(df["menopause"].sum()):,}   0: {int((df["menopause"]==0).sum()):,}')

RHQ305: {1.0: 244, 2.0: 2874, 9.0: 19, nan: 4300}
RHD043: {1.0: 17, 2.0: 5, 3.0: 448, 7.0: 751, 9.0: 150, 99.0: 4, nan: 6062}
RHQ031: {1.0: 1952, 2.0: 1386, 7.0: 2, 9.0: 1, nan: 4096}

menopause  →  1: 1,271   0: 6,166


---
## 14 · overweight

| NHANES variable | Renamed column | Question text |
|-----------------|----------------|---------------|
| MCQ080 | `mcq080___doctor_ever_said_you_were_overweight` | *"Has a doctor ever said you were overweight?"* |

**Formula:**
```
overweight = 1   if  MCQ080 == 1
           = 0   otherwise
```

**Notes:** This is a self-reported doctor diagnosis, not a BMI-derived flag. It captures individuals whose clinician explicitly flagged overweight as a concern.

In [15]:
col = 'mcq080___doctor_ever_said_you_were_overweight'
print(f'MCQ080: {df[col].value_counts(dropna=False).sort_index().to_dict()}')
print(f'overweight  →  1: {int(df["overweight"].sum()):,}   0: {int((df["overweight"]==0).sum()):,}')

MCQ080: {1.0: 2900, 2.0: 4528, 9.0: 9}
overweight  →  1: 2,900   0: 4,537


---
## 15 · alcohol

**Definition:** History of heavy daily alcohol use (≥ 4/5 drinks every day).

| NHANES variable | Renamed column | Question text |
|-----------------|----------------|---------------|
| ALQ151 | `alq151___ever_have_4/5_or_more_drinks_every_day?` | *"Was there ever a time you drank 4/5 or more drinks every day?"* |

**Formula:**
```
alcohol = 1   if  ALQ151 == 1
        = 0   otherwise
```

**ALQ151 encoding:** 1=Yes · 2=No · 7=Refused · 9=Don't know  
**Notes:** Captures lifetime heavy alcohol use pattern, not current use. Threshold is ≥5 drinks/day for men (≥4 for women) — standard NIAAA heavy-use criterion.

In [16]:
col = 'alq151___ever_have_4/5_or_more_drinks_every_day?'
print(f'ALQ151: {df[col].value_counts(dropna=False).sort_index().to_dict()}')
print(f'alcohol  →  1: {int(df["alcohol"].sum()):,}   0: {int((df["alcohol"]==0).sum()):,}')

ALQ151: {1.0: 851, 2.0: 4967, 9.0: 6, nan: 1613}
alcohol  →  1: 851   0: 6,586


---
## 16 · iron_deficiency

**Definition:** Serum ferritin below the lower limit of normal — iron stores depleted.

| NHANES variable | Lab column | Unit |
|-----------------|-----------|------|
| LBXFER | `LBXFER_ferritin_ng_ml` | ng/mL |

**Formula:**
```
iron_deficiency = 1   if  LBXFER < 15 ng/mL
                = 0   otherwise  (including NaN → 0)
```

**Threshold:** 15 ng/mL — WHO lower limit of normal for serum ferritin (iron stores depleted below this level regardless of haemoglobin).  
**Notes:** NaN (lab not collected) → 0. Does not require concurrent anaemia — iron deficiency without anaemia is clinically meaningful for fatigue.

In [17]:
col = 'LBXFER_ferritin_ng_ml'
s = df[col].dropna()
print(f'LBXFER  n={len(s):,}  min={s.min():.1f}  median={s.median():.1f}  max={s.max():.1f} ng/mL')
print(f'Below 15 ng/mL: {(s < 15).sum():,} of {len(s):,} ({(s<15).mean()*100:.1f}%)')
print(f'iron_deficiency  →  1: {int(df["iron_deficiency"].sum()):,}   0: {int((df["iron_deficiency"]==0).sum()):,}')

LBXFER  n=6,459  min=1.0  median=101.0  max=5190.0 ng/mL
Below 15 ng/mL: 450 of 6,459 (7.0%)
iron_deficiency  →  1: 450   0: 6,987


---
## 17 · hepatic_insufficiency

**Definition:** Elevated liver enzymes above upper limit of normal — indicating hepatocellular damage or biliary obstruction.

| NHANES variable | Lab column | Enzyme | ULN (M / F) |
|-----------------|-----------|--------|-------------|
| LBXSATSI | `LBXSATSI_alanine_aminotransferase_alt_u_l` | ALT | > 56 U/L (both) |
| LBXSASSI | `LBXSASSI_aspartate_aminotransferase_ast_u_l` | AST | > 40 U/L (both) |
| LBXSGTSI | `LBXSGTSI_gamma_glutamyl_transferase_ggt_iu_l` | GGT | > 61 U/L (M) / > 39 U/L (F) |

**Formula:**
```
hepatic_insufficiency = 1   if  LBXSATSI > 56
                             OR LBXSASSI > 40
                             OR ( gender == Male    AND LBXSGTSI > 61 )
                             OR ( gender == Female  AND LBXSGTSI > 39 )
                      = 0   otherwise  (including NaN → 0)
```

**Notes:** Thresholds are standard NHANES reference ranges. GGT sex-adjustment reflects physiological differences. Any single elevated enzyme is sufficient to flag the condition.

In [18]:
for col, thr, label in [
    ('LBXSATSI_alanine_aminotransferase_alt_u_l', 56,   'ALT > 56'),
    ('LBXSASSI_aspartate_aminotransferase_ast_u_l', 40, 'AST > 40'),
    ('LBXSGTSI_gamma_glutamyl_transferase_ggt_iu_l', None, 'GGT sex-adj'),
]:
    s = df[col].dropna()
    if thr:
        n = (s > thr).sum()
        print(f'  {label} U/L: {n:,} of {len(s):,} ({n/len(s)*100:.1f}%)')
    else:
        is_f = df.loc[df[col].notna(), 'gender'] == 'Female'
        n = ((df[col] > 61) & (df['gender']=='Male') | (df[col] > 39) & (df['gender']=='Female')).sum()
        print(f'  {label} (>61 M / >39 F): {n:,}')
print(f'\nhepatic_insufficiency  →  1: {int(df["hepatic_insufficiency"].sum()):,}   0: {int((df["hepatic_insufficiency"]==0).sum()):,}')

  ALT > 56 U/L: 266 of 6,387 (4.2%)
  AST > 40 U/L: 299 of 6,359 (4.7%)
  GGT sex-adj (>61 M / >39 F): 763

hepatic_insufficiency  →  1: 943   0: 6,494


---
## 18 · electrolyte_imbalance

**Definition:** Any major serum electrolyte outside its reference range.

| NHANES variable | Lab column | Electrolyte | Normal range |
|-----------------|-----------|------------|---------------|
| LBXSNASI | `LBXSNASI_sodium_mmol_l` | Sodium (Na⁺) | 136–145 mmol/L |
| LBXSKSI | `LBXSKSI_potassium_mmol_l` | Potassium (K⁺) | 3.5–5.0 mmol/L |
| LBXSCA | `LBXSCA_total_calcium_mg_dl` | Calcium (Ca²⁺) | 8.5–10.5 mg/dL |

**Formula:**
```
electrolyte_imbalance = 1   if  LBXSNASI < 136  OR  LBXSNASI > 145   # hypo/hypernatraemia
                             OR LBXSKSI  < 3.5  OR  LBXSKSI  > 5.0   # hypo/hyperkalaemia
                             OR LBXSCA   < 8.5  OR  LBXSCA   > 10.5  # hypo/hypercalcaemia
                      = 0   otherwise  (including NaN → 0)
```

In [19]:
checks = [
    ('LBXSNASI_sodium_mmol_l',    136, 145, 'Na mmol/L'),
    ('LBXSKSI_potassium_mmol_l',  3.5, 5.0, 'K  mmol/L'),
    ('LBXSCA_total_calcium_mg_dl', 8.5, 10.5,'Ca mg/dL'),
]
for col, lo, hi, label in checks:
    s = df[col].dropna()
    n_low  = (s < lo).sum()
    n_high = (s > hi).sum()
    print(f'  {label}  n={len(s):,}  <{lo}: {n_low}  >{hi}: {n_high}  total_out: {n_low+n_high}')
print(f'\nelectrolyte_imbalance  →  1: {int(df["electrolyte_imbalance"].sum()):,}   0: {int((df["electrolyte_imbalance"]==0).sum()):,}')

  Na mmol/L  n=6,390  <136: 167  >145: 127  total_out: 294
  K  mmol/L  n=6,383  <3.5: 190  >5.0: 31  total_out: 221
  Ca mg/dL  n=6,387  <8.5: 85  >10.5: 17  total_out: 102

electrolyte_imbalance  →  1: 579   0: 6,858


---
## 19 · infection_inflammation

**Definition:** Multi-marker scoring of chronic systemic inflammation using CRP, WBC, and NLR.
Acute infection cases (WBC > 15.0 **and** CRP > 10.0) are excluded — they represent acute infection, not chronic inflammation.

| NHANES variable | Lab column | Unit | Role |
|-----------------|-----------|------|------|
| LBXHSCRP | `LBXHSCRP_hs_c_reactive_protein_mg_l` | mg/L | CRP — primary inflammation marker |
| LBXWBCSI | `LBXWBCSI_white_blood_cell_count_1000_cells_ul` | 10³ cells/µL | WBC — immune activation |
| LBXNEPCT | `LBXNEPCT_segmented_neutrophils_percent` | % | Neutrophils % (for NLR) |
| LBXLYPCT | `LBXLYPCT_lymphocyte_percent` | % | Lymphocytes % (for NLR) |

**NLR:** `LBXNEPCT / LBXLYPCT` — Neutrophil-to-Lymphocyte Ratio (percentage-based)

**CRP reference ranges:** < 1.0 = low · 1–3 = borderline · > 3.0 = elevated · > 10.0 = acute/severe

**Scoring formula:**
```
CRP  > 10.0  → +3   (acute/severe)
CRP  >  3.0  → +2   (chronic elevated)
CRP  >  1.0  → +1   (borderline)

WBC  > 10.0  → +2   (leukocytosis)
WBC  >  7.5  → +1   (high-normal)

NLR  >  3.0  → +2   (chronic inflammation signal)
NLR  >  2.0  → +1   (borderline)

infection_inflammation = 1   if  score ≥ 3
                       = 0   if  score < 3
                       = NaN if  WBC > 15.0 AND CRP > 10.0   (acute_infection — excluded)
```

**Notes:**
- NaN scores for any individual marker are treated as 0 contribution (conservative).
- 9 cases flagged as `acute_infection` and excluded (set to NaN) to focus label on chronic inflammation.
- Previous definition used only `LBXWBCSI > 11.0` (simple leukocytosis threshold).

In [ ]:
import numpy as np

crp_col = 'LBXHSCRP_hs_c_reactive_protein_mg_l'
wbc_col = 'LBXWBCSI_white_blood_cell_count_1000_cells_ul'
nep_col = 'LBXNEPCT_segmented_neutrophils_percent'
lyp_col = 'LBXLYPCT_lymphocyte_percent'

nlr = df[nep_col] / df[lyp_col]

# Marker distributions
for col, label in [(crp_col,'CRP mg/L'), (wbc_col,'WBC 10³/µL'), ('nlr','NLR')]:
    s = nlr if col == 'nlr' else df[col]
    s = s.dropna()
    print(f'{label}  n={len(s):,}  median={s.median():.2f}  >threshold: ', end='')
    if col == crp_col:
        print(f'CRP>1={( s>1).sum():,}  CRP>3={(s>3).sum():,}  CRP>10={(s>10).sum():,}')
    elif col == wbc_col:
        print(f'WBC>7.5={(s>7.5).sum():,}  WBC>10={(s>10).sum():,}  WBC>15={(s>15).sum():,}')
    else:
        print(f'NLR>2={(s>2).sum():,}  NLR>3={(s>3).sum():,}')

# Acute exclusions
acute = ((df[wbc_col] > 15.0) & (df[crp_col] > 10.0))
print(f'\nAcute infection exclusions (WBC>15 & CRP>10): {acute.sum():,}')

n1   = int((df['infection_inflammation'] == 1).sum())
n0   = int((df['infection_inflammation'] == 0).sum())
n_na = int(df['infection_inflammation'].isna().sum())
print(f'\ninfection_inflammation  →  1: {n1:,}   0: {n0:,}   NaN(acute): {n_na:,}')
print(f'Prevalence (of scored cases): {n1/(n1+n0)*100:.1f}%')

---
## 20 · CFS_suspect

**Definition:** Suspect Chronic Fatigue Syndrome — operationalised from available NHANES variables as a proxy for the CDC/IOM ME/CFS criteria.

### Criteria applied

| Criterion | Variable | Condition | Rationale |
|-----------|----------|-----------|----------|
| Unrefreshing/disrupted sleep | SLQ040 | ≥ 3 (Frequently/Almost always) | Core ME/CFS symptom |
| Sustained healthcare engagement | HUQ030 | == 1 (Has regular care place) | Proxy for ongoing illness care-seeking |
| Persistent poor health | HUQ010 | ≥ 3 (Good/Fair/Poor) | Duration proxy — persistent impairment |
| Sleep apnoea ruled out | SLQ030 | ≤ 2 (Not heavy snorer) | Snoring excludes OSA as sleep cause |
| Not anaemic (lab) | LBXHGB | ≥ 12 g/dL (F) / ≥ 13.5 g/dL (M) | Exclude anaemia as fatigue cause |
| Not diabetic (lab) | LBXGH | < 6.5% | Exclude diabetes as fatigue cause |
| Not iron-deficient (lab) | LBXFER | ≥ 15 ng/mL | Exclude iron deficiency |
| No anemia treatment | MCQ053 | ≠ 1 | Exclude treated anaemia |
| No diabetes diagnosis | DIQ010 | ≠ 1 | Exclude diagnosed diabetes |
| No active thyroid disease | MCQ160M or MCQ170M | == 2 | Exclude thyroid as cause |
| No heart failure | MCQ160B | ≠ 1 | Exclude cardiac causes |
| No kidney disease | KIQ022 | ≠ 1 | Exclude renal causes |

### Criteria NOT available in this dataset

| Criterion | NHANES variable | Reason absent |
|-----------|----------------|---------------|
| Normal thyroid function (lab) | **LBXTSH1** (TSH) | Not merged into dataset |
| Activity limitation | **PFQ049** | PFQ module not merged |
| PHQ depression screen | **DPQ010–090 full screen** | Only DPQ040 (fatigue item) present |

**Formula:**
```python
CFS_suspect = 1   if ALL of:
    SLQ040 >= 3                             # frequent unrefreshing sleep
    AND HUQ030 == 1                         # has regular healthcare place
    AND HUQ010 >= 3                         # fair or poor general health
    AND SLQ030 <= 2                         # not a heavy snorer
    AND ( (gender==F AND LBXHGB >= 12.0)   # not anaemic
           OR (gender==M AND LBXHGB >= 13.5) )
    AND LBXGH  < 6.5                        # HbA1c normal
    AND LBXFER >= 15                        # ferritin normal
    AND MCQ053 != 1                         # no anemia treatment
    AND DIQ010 != 1                         # no diabetes
    AND (MCQ160M == 2 OR MCQ170M == 2)      # no active thyroid problem
    AND MCQ160B != 1                        # no heart failure
    AND KIQ022 != 1                         # no kidney disease
```

**NaN handling:** Any NaN in a required lab field → condition fails (conservative approach).  
**Expected prevalence:** Very low (< 1%) due to strict multi-criterion AND logic. Loosen `SLQ040 >= 3` → `>= 2` for a broader suspect screen.

In [21]:
cfs_criteria = {
    'SLQ040 >= 3':              (df['slq040___how_often_do_you_snort_or_stop_breathing'].replace({7:np.nan,9:np.nan}) >= 3),
    'HUQ030 == 1':              (df['huq030___routine_place_to_go_for_healthcare'] == 1),
    'HUQ010 >= 3':              (df['huq010___general_health_condition'].replace({7:np.nan,9:np.nan}) >= 3),
    'SLQ030 <= 2 (no heavy snore)': (df['slq030___how_often_do_you_snore?'].replace({7:np.nan,9:np.nan}) <= 2),
    'LBXHGB >= 12/13.5':        (
        ((df['gender']=='Female') & (df['LBXHGB_hemoglobin_g_dl'] >= 12.0)) |
        ((df['gender']=='Male')   & (df['LBXHGB_hemoglobin_g_dl'] >= 13.5))
    ),
    'LBXGH < 6.5%':             (df['LBXGH_glycohemoglobin'] < 6.5),
    'LBXFER >= 15':             (df['LBXFER_ferritin_ng_ml'] >= 15),
    'MCQ053 != 1':              (df['mcq053___taking_treatment_for_anemia/past_3_mos'] != 1),
    'DIQ010 != 1':              (df['diq010___doctor_told_you_have_diabetes'] != 1),
    'No active thyroid':        (
        (df['mcq160m___ever_told_you_had_thyroid_problem'] == 2) |
        (df['mcq170m___still_have_thyroid_problem'] == 2)
    ),
    'MCQ160B != 1':             (df['mcq160b___ever_told_you_had_congestive_heart_failure'] != 1),
    'KIQ022 != 1':              (df['kiq022___ever_told_you_had_weak/failing_kidneys?'] != 1),
}

print('Criterion pass rates (how many rows satisfy each individually):')
for label, mask in cfs_criteria.items():
    print(f'  {label:<38} {mask.fillna(False).sum():>5,}  ({mask.fillna(False).mean()*100:.1f}%)')

print(f'\nCFS_suspect  →  1: {int(df["CFS_suspect"].sum()):,}   0: {int((df["CFS_suspect"]==0).sum()):,}')
print('\n⚠️  Missing from dataset (TSH, PFQ049, full PHQ) make this a conservative proxy.')
print('   Consider relaxing SLQ040 >= 3 → >= 2 for a broader suspect screen.')

Criterion pass rates (how many rows satisfy each individually):
  SLQ040 >= 3                              380  (5.1%)
  HUQ030 == 1                            5,881  (79.1%)
  HUQ010 >= 3                            4,381  (58.9%)
  SLQ030 <= 2 (no heavy snore)           5,045  (67.8%)
  LBXHGB >= 12/13.5                      5,793  (77.9%)
  LBXGH < 6.5%                           5,823  (78.3%)
  LBXFER >= 15                           6,009  (80.8%)
  MCQ053 != 1                            7,079  (95.2%)
  DIQ010 != 1                            6,660  (89.6%)
  No active thyroid                      6,475  (87.1%)
  MCQ160B != 1                           7,288  (98.0%)
  KIQ022 != 1                            7,251  (97.5%)

CFS_suspect  →  1: 13   0: 7,424

⚠️  Missing from dataset (TSH, PFQ049, full PHQ) make this a conservative proxy.
   Consider relaxing SLQ040 >= 3 → >= 2 for a broader suspect screen.


---
## Supplementary · rxd_disease_list

**Source:** `P_RXQ_RX.xpt` — NHANES prescription medication file
**Column:** `rxd_disease_list` (added to final dataset)

Each person may appear multiple times in the raw file (once per medication).
`RXDRSD1`, `RXDRSD2`, `RXDRSD3` record up to 3 disease indications per medication.

**Derivation:**
```
1. Stack RXDRSD1/RXDRSD2/RXDRSD3 into long format (one row per disease mention)
2. Strip whitespace; drop empty / NaN values
3. Deduplicate per SEQN (unique diseases only)
4. Join with ", " separator → rxd_disease_list
5. Left-join onto final dataset on SEQN
```

**Coverage:** 3,529 of 7,437 adults have at least one RXDRSD entry.
Used directly as source for `myalgia`, `anxiety`, `depression`, and to enrich `sleep_disorder`.


In [22]:
# Coverage and sample entries
n_with_rxd = df['rxd_disease_list'].notna().sum()
print(f'Adults with rxd_disease_list populated: {n_with_rxd:,} / {len(df):,}')
print()
print('Sample disease lists (first 8 non-null):')
sample = df[df['rxd_disease_list'].notna()][['SEQN','rxd_disease_list']].head(8)
for _, row in sample.iterrows():
    diseases = row['rxd_disease_list']
    print(f'  SEQN {int(row["SEQN"])}: {diseases[:100]}{"..." if len(diseases)>100 else ""}')


Adults with rxd_disease_list populated: 3,529 / 7,437

Sample disease lists (first 8 non-null):
  SEQN 109268: Long term (current) use of hormonal contraceptives
  SEQN 109271: Epilepsy and recurrent seizures, Psoriasis, Pure hypercholesterolemia
  SEQN 109273: Functional dyspepsia, Heartburn, Other specified disorders of teeth and supporting structures, Peria...
  SEQN 109283: Chest pain, unspecified, Heart disease, unspecified, Heartburn, Myalgia, Pain in joint, Pure hyperch...
  SEQN 109286: Nausea and vomiting
  SEQN 109292: Allergy, unspecified, Dorsalgia, Essential (primary) hypertension, Insomnia, Neuralgia and neuritis,...
  SEQN 109295: Pain and other conditions associated with female genital organs and menstrual cycle, Sleep disorder,...
  SEQN 109313: Hypothyroidism, unspecified


---
## 21 · myalgia

**Definition:** Person takes medication prescribed for myalgia (muscle pain).

| Source | Column | Description |
|--------|--------|-------------|
| P_RXQ_RX RXDRSD1-3 | `rxd_disease_list` | Comma-separated medication disease indications |

**Formula:**
```
myalgia = 1   if  rxd_disease_list contains "Myalgia"   (case-insensitive)
        = 0   otherwise (including no RXD data)
```

**Notes:** Captures myalgia as a medication indication. Does not capture people with muscle pain who are not on relevant prescriptions. NHANES questionnaire contains some musculoskeletal pain questions but no dedicated myalgia column.


In [23]:
myalgia_raw = df['rxd_disease_list'].str.contains('Myalgia', case=False, na=False)
print(f'myalgia  ->  1: {int(df["myalgia"].sum()):,}   0: {int((df["myalgia"]==0).sum()):,}')
# Show sample disease lists for myalgia=1
print()
print('Sample rxd_disease_list entries for myalgia=1:')
print(df[df['myalgia']==1]['rxd_disease_list'].head(5).to_string())


myalgia  ->  1: 149   0: 7,288

Sample rxd_disease_list entries for myalgia=1:
5      Chest pain, unspecified, Heart disease, unspec...
67     Essential (primary) hypertension, Fibromyalgia...
83                            Myalgia, Pain, unspecified
134    Attention-deficit hyperactivity disorders, Dor...
136    Dorsalgia, Fibromyalgia, Gastro-esophageal ref...


---
## 22 · anxiety

**Definition:** Person takes medication prescribed for an anxiety disorder.

| Source | Column | Description |
|--------|--------|-------------|
| P_RXQ_RX RXDRSD1-3 | `rxd_disease_list` | Comma-separated medication disease indications |

**Formula:**
```
anxiety = 1   if  rxd_disease_list contains "Anxiety disorder"   (case-insensitive)
        = 0   otherwise (including no RXD data)
```

**Notes:** Matches the specific ICD-10 description "Anxiety disorder, unspecified" used in NHANES RXD coding. Broader anxiety-spectrum conditions (panic disorder, PTSD, OCD) are coded separately and are **not** captured by this label. People managing anxiety with non-prescription approaches are also not captured.


In [24]:
print(f'anxiety  ->  1: {int(df["anxiety"].sum()):,}   0: {int((df["anxiety"]==0).sum()):,}')
print()
print('Top RXD disease strings co-occurring with anxiety=1 (excluding "Anxiety disorder"):')
anx_lists = df[df['anxiety']==1]['rxd_disease_list'].dropna()
from collections import Counter
co = Counter()
for lst in anx_lists:
    for d in lst.split(', '):
        if 'Anxiety disorder' not in d:
            co[d] += 1
for disease, cnt in co.most_common(10):
    print(f'  {cnt:>4}  {disease}')


anxiety  ->  1: 488   0: 6,949

Top RXD disease strings co-occurring with anxiety=1 (excluding "Anxiety disorder"):
  1109  unspecified
   243  Major depressive disorder
   224  single episode
   180  Essential (primary) hypertension
   121  Pure hypercholesterolemia
    85  Gastro-esophageal reflux disease
    84  Insomnia
    68  Dorsalgia
    65  Type 2 diabetes mellitus
    45  Sleep disorder


---
## 23 · depression

**Definition:** Person takes medication prescribed for a depressive disorder.

| Source | Column | Description |
|--------|--------|-------------|
| P_RXQ_RX RXDRSD1-3 | `rxd_disease_list` | Comma-separated medication disease indications |

**Formula:**
```
depression = 1   if  rxd_disease_list contains "depressive disorder"   (case-insensitive)
           = 0   otherwise (including no RXD data)
```

**Notes:** Matches any string containing "depressive disorder" — covers both
*"Major depressive disorder, single episode, unspecified"* (n≈695 in pivot) and
*"Major depressive disorder, recurrent, unspecified"* (n≈39).
People managing depression without medication are not captured. PHQ-9 questionnaire items (DPQ010–DPQ090) in the dataset offer an alternative continuous depression severity measure.


In [25]:
print(f'depression  ->  1: {int(df["depression"].sum()):,}   0: {int((df["depression"]==0).sum()):,}')
print()
# Break down by which depressive disorder subtype
dep_lists = df[df['depression']==1]['rxd_disease_list'].dropna()
from collections import Counter
subtypes = Counter()
for lst in dep_lists:
    for d in lst.split(', '):
        if 'depressive' in d.lower():
            subtypes[d] += 1
print('Depression subtypes found in rxd_disease_list:')
for subtype, cnt in subtypes.most_common():
    print(f'  {cnt:>4}  {subtype}')


depression  ->  1: 477   0: 6,960

Depression subtypes found in rxd_disease_list:
   493  Major depressive disorder


---
## Summary — All 23 Disease Labels


In [26]:
summary = []
for col in ALL_23:
    n1  = int(df[col].sum())
    n0  = int((df[col] == 0).sum())
    pct = n1 / (n0 + n1) * 100
    if col in ['iron_deficiency','hepatic_insufficiency',
               'electrolyte_imbalance','infection_inflammation']:
        typ = 'Lab'
    elif col in ['kidney','CFS_suspect']:
        typ = 'Q+Lab'
    elif col in ['sleep_disorder']:
        typ = 'Q+RXD'
    elif col in ['myalgia','anxiety','depression']:
        typ = 'RXD'
    else:
        typ = 'Questionnaire'
    summary.append({'disease': col, 'type': typ, 'n_positive': n1,
                    'n_negative': n0, 'prevalence_%': round(pct, 1)})

summary_df = pd.DataFrame(summary).sort_values('n_positive', ascending=False).reset_index(drop=True)
summary_df.index += 1
print(summary_df.to_string())


                   disease           type  n_positive  n_negative  prevalence_%
1               overweight  Questionnaire        2900        4537          39.0
2           sleep_disorder          Q+RXD        2403        5034          32.3
3      high_blood_pressure  Questionnaire        2130        5307          28.6
4         high_cholesterol  Questionnaire        1977        5460          26.6
5                 diabetes  Questionnaire        1645        5792          22.1
6                menopause  Questionnaire        1271        6166          17.1
7    hepatic_insufficiency            Lab         943        6494          12.7
8                  alcohol  Questionnaire         851        6586          11.4
9    electrolyte_imbalance            Lab         579        6858           7.8
10                 anxiety            RXD         488        6949           6.6
11              depression            RXD         477        6960           6.4
12         emphysema_lungs  Questionnair